# Join & Assemble: `bn_ready_baseline.csv`
**Integration step**: assembles the 5 preprocessed blocks into the
single cross-modality scaffold the group hands to bnlearn (R).

Inputs (all `data/processed/*_baseline_wide.csv`, all keyed on `subject.subjectGuid` +
`sample.sampleKitGuid`):
- `clinical`  — the spine (92 samples; canonical metadata + 3 discrete roots + clinical nodes)
- `olink` (92), `celltype_freq` (92), `pseudobulk` (92), `wholeblood` (91)

Output: `data/processed/bn_ready_baseline.csv`

**Design decisions:**
- **Left-join each molecular block onto the spine** on `sample.sampleKitGuid`. The spine is a
  strict superset (every molecular kit is already in it — 0 orphans), so this equals an outer
  join here but pins the **92-row** invariant and keeps spine metadata authoritative. Each
  block's redundant `subject.subjectGuid` is dropped (verified 100% consistent per kit).
- **Full assembly** — carry every feature + flag column (~868 cols, p>>n). Node *selection*
  stays downstream in R (flat-prior discipline; owners defer to BN-time). **No slim table**
  yet: pseudobulk/whole-blood haven't committed subsets, and `bn_node_tiers.csv` lacks clinical.
- **Coverage columns** (`coverage.*`) describe per-sample block presence (no exclusion decision
  baked in) to feed the group's later complete-case step.
- **Column order:** keys -> spine metadata/roots -> clinical features -> molecular features ->
  coverage -> all `*_flag*` last.
- **No imputation** — structured missingness (the 1 whole-blood-missing sample) left as NaN,
  reported. **Validation (`val.*`) excluded** — held-out, built + joined separately at benchmarking.

## Stage 0 — Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))
from src.load import load_csv

PROCESSED = Path("../data/processed")

SUBJ_KEY = "subject.subjectGuid"
SAMPLE_KEY = "sample.sampleKitGuid"
KEYS = [SUBJ_KEY, SAMPLE_KEY]

SPINE = "clinical"
MOLECULAR = ["olink", "celltype_freq", "pseudobulk", "wholeblood"]  # fixed join/column order
ALL_BLOCKS = [SPINE] + MOLECULAR

def is_flag(c: str) -> bool:
    """Flag column across all block conventions (_flag_, _low_ncells_flag, _low_lib_flag)."""
    return ("_flag_" in c) or c.endswith("_flag")

OUT = PROCESSED / "bn_ready_baseline.csv" 

## Stage 1 — Load Blocks

In [2]:
blocks = {}
for b in ALL_BLOCKS:
    df = load_csv(PROCESSED / f"{b}_baseline_wide.csv")
    blocks[b] = df
    nfeat = sum(1 for c in df.columns if c not in KEYS and not is_flag(c))
    nflag = sum(1 for c in df.columns if is_flag(c))
    print(f"{b:14s} shape={str(df.shape):11s} samples={df[SAMPLE_KEY].nunique():3d} "
          f"features={nfeat:3d} flags={nflag:3d}")

spine = blocks[SPINE]
print(f"\nspine (anchor): {spine.shape[0]} baseline samples")

clinical       shape=(92, 163)   samples= 92 features= 69 flags= 92
olink          shape=(92, 75)    samples= 92 features= 35 flags= 38
celltype_freq  shape=(92, 123)   samples= 92 features=121 flags=  0
pseudobulk     shape=(92, 457)   samples= 92 features=390 flags= 65
wholeblood     shape=(91, 53)    samples= 91 features= 50 flags=  1

spine (anchor): 92 baseline samples


## Stage 2 — Integrity Checks (before joining)

Guarantee the join is a clean left-merge: no duplicate kits, every molecular kit already in the
spine (no orphans -> left == outer, no row growth), and `subject.subjectGuid` consistent per kit
so dropping the redundant copy is safe.

In [3]:
spine_kits = set(spine[SAMPLE_KEY])

for b in ALL_BLOCKS:
    df = blocks[b]
    assert not df[SAMPLE_KEY].duplicated().any(), f"{b}: duplicate sampleKitGuid"

for b in MOLECULAR:
    kits = set(blocks[b][SAMPLE_KEY])
    orphans = kits - spine_kits
    assert not orphans, f"{b}: {len(orphans)} kit(s) not in spine -> would grow rows: {list(orphans)[:5]}"
    # subject.subjectGuid consistency vs spine
    chk = (blocks[b][[SAMPLE_KEY, SUBJ_KEY]]
           .merge(spine[[SAMPLE_KEY, SUBJ_KEY]], on=SAMPLE_KEY, suffixes=("_blk", "_spine")))
    mism = int((chk[f"{SUBJ_KEY}_blk"] != chk[f"{SUBJ_KEY}_spine"]).sum())
    assert mism == 0, f"{b}: {mism} subjectGuid mismatch vs spine"
    print(f"{b:14s} kits in spine: {len(kits)}/{len(spine_kits)}  orphans: 0  subjGuid mismatches: 0")
print("\nall integrity checks passed -> safe left-join, no row growth.")

olink          kits in spine: 92/92  orphans: 0  subjGuid mismatches: 0
celltype_freq  kits in spine: 92/92  orphans: 0  subjGuid mismatches: 0
pseudobulk     kits in spine: 92/92  orphans: 0  subjGuid mismatches: 0
wholeblood     kits in spine: 91/92  orphans: 0  subjGuid mismatches: 0

all integrity checks passed -> safe left-join, no row growth.


## Stage 3 — Left-Join Molecular Blocks onto the Spine

In [4]:
merged = spine.copy()
for b in MOLECULAR:
    right = blocks[b].drop(columns=[SUBJ_KEY])   # drop redundant key; join on sampleKit only
    before = merged.shape
    merged = merged.merge(right, on=SAMPLE_KEY, how="left", validate="one_to_one")
    print(f"+ {b:14s} {before} -> {merged.shape}")

assert merged.shape[0] == 92, f"row count changed: {merged.shape[0]} (expected 92)"
assert not merged[SAMPLE_KEY].duplicated().any(), "duplicate kits after join"
print(f"\nmerged: {merged.shape}  (rows pinned at 92)")

+ olink          (92, 163) -> (92, 236)
+ celltype_freq  (92, 236) -> (92, 357)
+ pseudobulk     (92, 357) -> (92, 812)


+ wholeblood     (92, 812) -> (92, 863)

merged: (92, 863)  (rows pinned at 92)


## Stage 4 — Cross-Modality Coverage

`coverage.<block>` = this sample has that molecular block (describes presence only — no exclusion
decision). The clinical spine is present for every row by construction, so coverage is tracked
for the 4 molecular blocks.

In [5]:
for b in MOLECULAR:
    kits = set(blocks[b][SAMPLE_KEY])
    merged[f"coverage.{b}"] = merged[SAMPLE_KEY].isin(kits)

cov_cols = [f"coverage.{b}" for b in MOLECULAR]
merged["coverage.n_molecular"] = merged[cov_cols].sum(axis=1).astype(int)
merged["coverage.all_molecular"] = merged["coverage.n_molecular"] == len(MOLECULAR)

n_all = int(merged["coverage.all_molecular"].sum())
print(f"samples with ALL {len(MOLECULAR)} molecular blocks: {n_all}/{len(merged)}")
print(f"samples with partial coverage:                  {len(merged) - n_all}")
print("\nper-block coverage:")
for b in MOLECULAR:
    print(f"  {b:14s} {int(merged[f'coverage.{b}'].sum()):3d}/{len(merged)}")
print("\npartial-coverage samples (kept, NaN in missing block — no imputation):")
partial = merged.loc[~merged["coverage.all_molecular"], [SUBJ_KEY, SAMPLE_KEY] + cov_cols]
print(partial.to_string(index=False) if len(partial) else "  none")

samples with ALL 4 molecular blocks: 91/92
samples with partial coverage:                  1

per-block coverage:
  olink           92/92
  celltype_freq   92/92
  pseudobulk      92/92
  wholeblood      91/92

partial-coverage samples (kept, NaN in missing block — no imputation):
subject.subjectGuid sample.sampleKitGuid  coverage.olink  coverage.celltype_freq  coverage.pseudobulk  coverage.wholeblood
             BR1021              KT00338            True                    True                 True                False


## Stage 5 — Column Ordering

keys -> spine metadata/roots/context -> `clinical.*` features -> molecular features
(olink, freq, pb, wb) -> `coverage.*` -> all `*_flag*` columns last.

In [6]:
cov_all = [f"coverage.{b}" for b in MOLECULAR] + ["coverage.n_molecular", "coverage.all_molecular"]

spine_meta = [c for c in spine.columns
              if c not in KEYS and not c.startswith("clinical.") and not is_flag(c)]
clinical_feat = sorted(c for c in spine.columns if c.startswith("clinical.") and not is_flag(c))

mol_feat = []
for b in MOLECULAR:
    mol_feat += [c for c in blocks[b].columns if c not in KEYS and not is_flag(c)]

flag_cols = []
for b in ALL_BLOCKS:
    flag_cols += [c for c in blocks[b].columns if is_flag(c)]

order = KEYS + spine_meta + clinical_feat + mol_feat + cov_all + flag_cols

assert len(order) == len(set(order)), "duplicate column in order"
assert set(order) == set(merged.columns), (
    f"ordering mismatch: missing={set(merged.columns) - set(order)} "
    f"extra={set(order) - set(merged.columns)}")
merged = merged[order]
print(f"columns ordered: {len(order)}")
print(f"  keys={len(KEYS)}  spine_meta={len(spine_meta)}  clinical_feat={len(clinical_feat)}")
print(f"  molecular_feat={len(mol_feat)}  coverage={len(cov_all)}  flags_last={len(flag_cols)}")

columns ordered: 869
  keys=2  spine_meta=16  clinical_feat=53
  molecular_feat=596  coverage=6  flags_last=196


## Stage 6 — Validate & Export

In [7]:
# Inline contract (mixed prefixes -> single-prefix enforce_export_contract N/A).
assert list(merged.columns[:2]) == KEYS, "identifier columns not first"
assert merged[SAMPLE_KEY].is_unique, "sampleKitGuid not unique"
assert merged[KEYS].notna().all().all(), "null identifier"
assert merged.shape[0] == 92, "row count != 92"
for c in [SUBJ_KEY, SAMPLE_KEY, "subject.ageGroup", "subject.biologicalSex",
          "cmv.igg_serology_interpretation"]:
    assert c in merged.columns, f"missing spine column: {c}"
# every source feature/flag column survived the assembly
expected = set(KEYS) | set(cov_all)
for b in ALL_BLOCKS:
    expected |= {c for c in blocks[b].columns if c != SUBJ_KEY or b == SPINE}
assert set(merged.columns) == expected, (
    f"column accounting drift: {set(merged.columns) ^ expected}")
print("inline export contract passed.")

merged.to_csv(OUT, index=False)
print(f"written: {OUT}  ({merged.shape[0]} x {merged.shape[1]})")

inline export contract passed.


written: ..\data\processed\bn_ready_baseline.csv  (92 x 869)


In [8]:
# --- hand-off summary ---
n_flags = sum(1 for c in merged.columns if is_flag(c))
n_nodes = len(clinical_feat) + 3 + len(mol_feat)  # clinical + 3 roots + molecular features
print("=== bn_ready_baseline.csv HAND-OFF ===")
print(f"rows (baseline samples):   {merged.shape[0]}")
print(f"cols (total):              {merged.shape[1]}")
print(f"  candidate node columns:  ~{n_nodes} (3 roots + {len(clinical_feat)} clinical + {len(mol_feat)} molecular)")
print(f"  coverage columns:        {len(cov_all)}")
print(f"  flag columns (last):     {n_flags}")
print(f"join keys:                 {SUBJ_KEY}, {SAMPLE_KEY}")
print(f"complete (all 5 blocks):   {int(merged['coverage.all_molecular'].sum())}/{len(merged)}")
print(f"output:                    {OUT}")
print("\nNOTE: full scaffold (p>>n). Node selection + complete-case + discretization are")
print("downstream (R / group). val.* validation block joined separately at benchmarking.")
merged.iloc[:5, :8]

=== bn_ready_baseline.csv HAND-OFF ===
rows (baseline samples):   92
cols (total):              869
  candidate node columns:  ~652 (3 roots + 53 clinical + 596 molecular)
  coverage columns:        6
  flag columns (last):     196
join keys:                 subject.subjectGuid, sample.sampleKitGuid
complete (all 5 blocks):   91/92
output:                    ..\data\processed\bn_ready_baseline.csv

NOTE: full scaffold (p>>n). Node selection + complete-case + discretization are
downstream (R / group). val.* validation block joined separately at benchmarking.


,subject.subjectGuid,sample.sampleKitGuid,cohort.cohortGuid,subject.biologicalSex,subject.birthYear,subject.ageAtFirstDraw,subject.ageGroup,subject.race
0,BR1001,KT00001,BR1,Female,1987,32,Young Adult,Caucasian
1,BR1002,KT00002,BR1,Male,1991,28,Young Adult,Caucasian
2,BR1003,KT00003,BR1,Female,1989,30,Young Adult,Caucasian
3,BR1004,KT00004,BR1,Male,1989,30,Young Adult,Caucasian
4,BR1005,KT00006,BR1,Female,1992,27,Young Adult,Caucasian
